# TripoSR server (Colab, free T4 GPU)

Self-hosts TripoSR and exposes it as an API for `backend/main.py` to call.
Runtime -> Change runtime type -> T4 GPU before running.

Needs a free ngrok authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
— paste it in the last cell.

After the first cell: `Runtime` -> `Restart session`, then run cell 1
again followed by the rest. (Colab ships numpy2-linked cupy/scipy;
TripoSR needs numpy<2, so the first run leaves things in a broken
state until restarted once.)

In [ ]:
!git clone https://github.com/VAST-AI-Research/TripoSR
%cd TripoSR
!pip install -r requirements.txt -q
!pip install fastapi uvicorn python-multipart pyngrok nest-asyncio onnxruntime -q
# TripoSR downgrades numpy below 2.0, which breaks Colab's pre-installed
# cupy and scipy (both built against numpy>=2.0). cupy isn't needed at all;
# scipy is needed, so reinstall it to match instead of removing it.
!pip uninstall -y cupy-cuda12x -q
!pip install --force-reinstall --no-cache-dir "numpy==1.26.4" scipy -q
print("Install done. Now go to Runtime -> Restart session, then run this")
print("cell again (it's fast now, everything's cached) followed by the rest.")

In [ ]:
import torch
from tsr.system import TSR

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.renderer.set_chunk_size(8192)
model.to(device)
print("Model loaded.")

In [ ]:
import numpy as np
from PIL import Image
import rembg
from tsr.utils import remove_background, resize_foreground

rembg_session = rembg.new_session()

def process_image(image: Image.Image) -> Image.Image:
    image = remove_background(image, rembg_session)
    image = resize_foreground(image, 0.85)
    arr = np.array(image).astype(np.float32) / 255.0
    arr = arr[:, :, :3] * arr[:, :, 3:4] + (1 - arr[:, :, 3:4]) * 0.5
    return Image.fromarray((arr * 255.0).astype(np.uint8))

def generate_glb(image: Image.Image, out_path: str, resolution: int = 256):
    processed = process_image(image)
    with torch.no_grad():
        scene_codes = model([processed], device=device)
    mesh = model.extract_mesh(scene_codes, has_vertex_color=True, resolution=resolution)[0]
    mesh.export(out_path)
    torch.cuda.empty_cache()
    return out_path

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import uuid, os, io

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

OUT_DIR = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

@app.post("/convert")
async def convert(file: UploadFile = File(...)):
    img = Image.open(io.BytesIO(await file.read())).convert("RGBA")
    out_path = f"{OUT_DIR}/{uuid.uuid4()}.glb"
    generate_glb(img, out_path)
    return FileResponse(out_path, media_type="model/gltf-binary", filename="model.glb")

@app.get("/health")
def health():
    return {"status": "ok", "device": device}

In [ ]:
# Paste your free ngrok authtoken between the quotes below, then run this cell.
# Get one at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "PASTE_YOUR_TOKEN_HERE"

import uvicorn
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
public_url = ngrok.connect(8000)
print("\n>>> COPY THIS URL into COLAB_SERVER_URL in your backend/main.py:")
print(public_url)
print("\nLeave this cell running — closing it stops the server.\n")

config = uvicorn.Config(app, port=8000)
server = uvicorn.Server(config)
await server.serve()